# Dunnhumby 마지막 주 seed49 — M4-A 단독

기존 M1·M3·M4-B·M5-B는 SHA로 고정해 재사용하고, **binary graph + 원형 q_C 행가중 M4-A**만 300 epoch 학습합니다. 마지막 1주 test는 이미 노출된 단일 seed 예비 판독이며, validation·holdout·조기종료를 사용하지 않습니다.


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import drive
try:
    mounted = Path('/content/drive/MyDrive').is_dir()
except OSError:
    mounted = False
if not mounted:
    try:
        drive.mount('/content/drive')
    except (ValueError, OSError):
        subprocess.run(['fusermount', '-u', '/content/drive'], check=False)
        drive.mount('/content/drive', force_remount=True)


In [ ]:
SOURCE_COMMIT = '12e37a2b7139fa0dd1dfd0166cd441b5ce0c9ca1'
REPO = Path('/content/clv-dh-m4a-lastweek')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
print(torch.cuda.get_device_name(0))


In [ ]:
import clv_dh_m4a_m5a_lastweek_completion as screen
OUT_DIR = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m4a_m5a_lastweek_seed49_v1'
cfg = screen.configure(out_dir=OUT_DIR)
prepared = screen.prepare(cfg)
print({'train_end': 704, 'test': '705~711', 'seed': 49, 'epochs': 300, 'model': screen.M4_A})


In [ ]:
row = screen.run_selected(cfg, prepared, screen.M4_A)
print('완료:', row['model_id'], row['epochs'])


In [ ]:
result = screen.report_if_complete(cfg, prepared)
if not result['complete']:
    print(result['message'])
    print('미완료:', result['missing'])
else:
    from IPython.display import display
    key = ['model_id','recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
    display(result['absolute'][key])
    print(result['reading'])


In [ ]:
# 두 A arm이 모두 완료된 뒤에만 결과 ZIP을 생성합니다. 아직이면 이 셀만 다시 실행하세요.
result = screen.report_if_complete(cfg, prepared)
if result['complete']:
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files
    output = Path('/content/dh_m4a_m5a_lastweek_seed49_results.zip')
    with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
        for path in result['paths'].values():
            archive.write(path, Path(path).name)
        for name in ('protocol.json','reference_audit.json'):
            path = prepared['run_dir'] / name
            archive.write(path, name)
    files.download(str(output))
else:
    print(result['message'], result['missing'])
